# 03 — Exploratory Data Analysis
## EV Population Dataset — Washington State

**Project:** Data Visualisation & Analytics — Team Project  
**Notebook:** 03 of 05  
**Objective:** Understand the structure, distributions, trends, and outliers in the cleaned EV registration dataset before modelling and KPI computation.

---

### Table of Contents
1. [Setup & Data Load](#1)
2. [Dataset Overview](#2)
3. [Univariate Analysis — Numeric Columns](#3)
4. [Univariate Analysis — Categorical Columns](#4)
5. [Bivariate & Group Analysis](#5)
6. [Trend Analysis — Model Year](#6)
7. [Geographic Concentration](#7)
8. [Outlier Detection](#8)
9. [Key EDA Findings Summary](#9)

---
## 1. Setup & Data Load <a id='1'></a>

In [ ]:
import numpy as np
import pandas as pd

# Display settings
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.width', 120)

# Load cleaned dataset
DATA_PATH = '../data/EV_Population_Cleaned.csv'
df = pd.read_csv(DATA_PATH)

print(f'Dataset loaded successfully — {df.shape[0]:,} rows × {df.shape[1]} columns')

---
## 2. Dataset Overview <a id='2'></a>

In [ ]:
# ----- 2.1 Shape & Column Inventory -----
print('=== Shape ===')
print(f'Rows   : {df.shape[0]:,}')
print(f'Columns: {df.shape[1]}')
print()

print('=== Column Names & Data Types ===')
print(df.dtypes.to_string())

In [ ]:
# ----- 2.2 First 5 Rows -----
df.head()

In [ ]:
# ----- 2.3 Missing Value Audit -----
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    'Missing Count': missing,
    'Missing %'    : missing_pct
})

print('=== Missing Value Report ===')
print(missing_report[missing_report['Missing Count'] > 0])
print()
print(f'Columns with zero missing: {(missing == 0).sum()} / {df.shape[1]}')

In [ ]:
# ----- 2.4 Duplicate Check -----
dup_count = df.duplicated().sum()
print(f'Duplicate rows: {dup_count}')

# VIN should be unique per registration
vin_dupes = df['VIN (1-10)'].duplicated().sum()
print(f'Duplicate VINs: {vin_dupes}')
print('Note: Partial VINs (first 10 chars) may collide — DOL Vehicle ID is the true unique key.')

dol_dupes = df['DOL Vehicle ID'].duplicated().sum()
print(f'Duplicate DOL Vehicle IDs: {dol_dupes}')

In [ ]:
# ----- 2.5 Numeric Summary Statistics -----
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print('Numeric columns:', numeric_cols)
print()
df[numeric_cols].describe().T

In [ ]:
# ----- 2.6 Categorical Summary -----
cat_cols = df.select_dtypes(include='object').columns.tolist()
print('Categorical columns:', cat_cols)
print()

cat_summary = pd.DataFrame({
    'Unique Values': [df[c].nunique() for c in cat_cols],
    'Top Value'    : [df[c].value_counts().idxmax() for c in cat_cols],
    'Top Frequency': [df[c].value_counts().max() for c in cat_cols],
    'Top %'        : [(df[c].value_counts().max() / len(df) * 100).round(1) for c in cat_cols]
}, index=cat_cols)

print(cat_summary.to_string())

---
## 3. Univariate Analysis — Numeric Columns <a id='3'></a>

In [ ]:
# ----- 3.1 Electric Range — Core Distribution -----
er = df['Electric Range']

print('=== Electric Range — Descriptive Statistics ===')
print(f'  Count  : {er.count():,}')
print(f'  Mean   : {er.mean():.1f} miles')
print(f'  Median : {er.median():.1f} miles')
print(f'  Std Dev: {er.std():.1f} miles')
print(f'  Min    : {er.min():.0f} miles')
print(f'  Max    : {er.max():.0f} miles')
print()

print('=== Percentile Breakdown ===')
pcts = [10, 25, 50, 75, 90, 95, 99]
for p in pcts:
    print(f'  P{p:2d}: {np.percentile(er, p):.1f} miles')
print()

# Skewness and kurtosis using numpy
mean = er.mean()
std  = er.std()
skew = ((er - mean)**3).mean() / std**3
kurt = ((er - mean)**4).mean() / std**4 - 3
print(f'  Skewness : {skew:.3f}  (positive → right-skewed)')
print(f'  Kurtosis : {kurt:.3f}  (excess kurtosis; >0 = heavy tails)')

In [ ]:
# ----- 3.2 Electric Range — Frequency Binning -----
bins   = [0, 30, 60, 100, 150, 200, 250, 300, 400]
labels = ['0–30', '31–60', '61–100', '101–150', '151–200', '201–250', '251–300', '301–400']

df['Range_Bin'] = pd.cut(df['Electric Range'], bins=bins, labels=labels, right=True)
range_freq = df['Range_Bin'].value_counts().sort_index()

print('=== Electric Range — Frequency Distribution by Bin ===')
for label, count in range_freq.items():
    bar = '█' * (count // 500)
    print(f'  {label:>10} miles : {count:>6,}  {bar}')

In [ ]:
# ----- 3.3 Model Year Distribution -----
print('=== Model Year — Registration Counts ===')
yr_counts = df['Model Year'].value_counts().sort_index()

for yr, cnt in yr_counts.items():
    bar = '█' * (cnt // 300)
    print(f'  {yr}: {cnt:>6,}  {bar}')

print()
print(f'  Earliest registration year : {df["Model Year"].min()}')
print(f'  Latest  registration year  : {df["Model Year"].max()}')
print(f'  Most common year           : {df["Model Year"].mode()[0]}')

In [ ]:
# ----- 3.4 Vehicle Age Distribution -----
va = df['Vehicle Age']

print('=== Vehicle Age — Descriptive Statistics ===')
print(f'  Mean   : {va.mean():.2f} years')
print(f'  Median : {va.median():.1f} years')
print(f'  Std Dev: {va.std():.2f} years')
print(f'  Min    : {va.min():.0f} years')
print(f'  Max    : {va.max():.0f} years')
print()

print('=== Vehicle Age — Frequency Counts ===')
age_counts = va.value_counts().sort_index()
for age, cnt in age_counts.items():
    bar = '█' * (cnt // 500)
    print(f'  Age {age:>2}: {cnt:>6,}  {bar}')

In [ ]:
# ----- 3.5 Utility Complexity Score -----
print('=== Utility Complexity Score — Distribution ===')
ucs = df['Utility_Complexity_Score'].value_counts().sort_index()
for score, cnt in ucs.items():
    pct = cnt / len(df) * 100
    bar = '█' * (cnt // 1000)
    print(f'  Score {score}: {cnt:>6,}  ({pct:.1f}%)  {bar}')

print()
print(f'  Mean score  : {df["Utility_Complexity_Score"].mean():.3f}')
print(f'  Std Dev     : {df["Utility_Complexity_Score"].std():.3f}')

---
## 4. Univariate Analysis — Categorical Columns <a id='4'></a>

In [ ]:
# ----- 4.1 Electric Vehicle Type -----
print('=== Electric Vehicle Type ===')
ev_type = df['Electric Vehicle Type'].value_counts()
for t, cnt in ev_type.items():
    short = 'BEV' if 'Battery' in t else 'PHEV'
    pct   = cnt / len(df) * 100
    print(f'  {short} ({t}) : {cnt:,}  ({pct:.1f}%)')

In [ ]:
# ----- 4.2 Top 15 Makes -----
print('=== Top 15 Makes by Registration Count ===')
top_makes = df['Make'].value_counts().head(15)

for make, cnt in top_makes.items():
    pct = cnt / len(df) * 100
    bar = '█' * (cnt // 500)
    print(f'  {make:<15}: {cnt:>6,}  ({pct:4.1f}%)  {bar}')

print()
print(f'  Total unique makes: {df["Make"].nunique()}')
print(f'  Top 5 makes cover : {top_makes.head(5).sum() / len(df) * 100:.1f}% of fleet')

In [ ]:
# ----- 4.3 Tesla Model Breakdown -----
tesla = df[df['Make'] == 'TESLA']
print(f'=== Tesla Models ({len(tesla):,} registrations) ===')
for model, cnt in tesla['Model'].value_counts().items():
    pct = cnt / len(tesla) * 100
    print(f'  {model:<12}: {cnt:>6,}  ({pct:.1f}%)')

In [ ]:
# ----- 4.4 Market Segment -----
print('=== Market Segment ===')
for seg, cnt in df['Market_Segment'].value_counts().items():
    pct = cnt / len(df) * 100
    print(f'  {seg:<12}: {cnt:>6,}  ({pct:.1f}%)')

In [ ]:
# ----- 4.5 Adoption Wave -----
print('=== Adoption Wave ===')
wave_order = ['Early', 'Growth', 'Mass Adoption']
for wave in wave_order:
    cnt = (df['Adoption_Wave'] == wave).sum()
    pct = cnt / len(df) * 100
    bar = '█' * (cnt // 1000)
    print(f'  {wave:<14}: {cnt:>6,}  ({pct:.1f}%)  {bar}')

In [ ]:
# ----- 4.6 Range Category -----
print('=== Range Category ===')
for cat, cnt in df['Range_Category'].value_counts().items():
    pct = cnt / len(df) * 100
    bar = '█' * (cnt // 1000)
    print(f'  {cat:<18}: {cnt:>6,}  ({pct:.1f}%)  {bar}')

In [ ]:
# ----- 4.7 CAFV Eligibility -----
print('=== CAFV Status ===')
for status, cnt in df['CAFV_Status'].value_counts().items():
    pct = cnt / len(df) * 100
    print(f'  {status:<25}: {cnt:>6,}  ({pct:.1f}%)')

---
## 5. Bivariate & Group Analysis <a id='5'></a>

In [ ]:
# ----- 5.1 Electric Range by Vehicle Type -----
print('=== Average Electric Range by EV Type ===')
range_by_type = df.groupby('Electric Vehicle Type')['Electric Range'].agg(
    Count='count', Mean='mean', Median='median', Std='std', Min='min', Max='max'
).round(2)
print(range_by_type.to_string())

In [ ]:
# ----- 5.2 Electric Range by Top 10 Makes -----
top10_makes = df['Make'].value_counts().head(10).index

print('=== Electric Range Stats — Top 10 Makes ===')
range_by_make = (
    df[df['Make'].isin(top10_makes)]
    .groupby('Make')['Electric Range']
    .agg(Count='count', Mean='mean', Median='median', Std='std')
    .round(1)
    .sort_values('Mean', ascending=False)
)
print(range_by_make.to_string())

In [ ]:
# ----- 5.3 Adoption Wave × EV Type Cross-tab -----
print('=== Adoption Wave × EV Type — Counts ===')
ct = pd.crosstab(df['Adoption_Wave'], df['Electric Vehicle Type'], margins=True)
print(ct.to_string())
print()

print('=== Adoption Wave × EV Type — Row % ===')
ct_pct = pd.crosstab(df['Adoption_Wave'], df['Electric Vehicle Type'], normalize='index').mul(100).round(1)
print(ct_pct.to_string())

In [ ]:
# ----- 5.4 Market Segment × EV Type -----
print('=== Market Segment × EV Type — Counts ===')
ct2 = pd.crosstab(df['Market_Segment'], df['Electric Vehicle Type'], margins=True)
print(ct2.to_string())
print()

print('=== Market Segment × EV Type — Row % ===')
ct2_pct = pd.crosstab(df['Market_Segment'], df['Electric Vehicle Type'], normalize='index').mul(100).round(1)
print(ct2_pct.to_string())

In [ ]:
# ----- 5.5 CAFV Eligibility × EV Type -----
print('=== CAFV Eligibility × EV Type ===')
ct3 = pd.crosstab(df['CAFV_Status'], df['Electric Vehicle Type'], normalize='index').mul(100).round(1)
print(ct3.to_string())

In [ ]:
# ----- 5.6 Vehicle Age vs Electric Range — Correlation -----
# Using numpy corrcoef
corr_age_range = np.corrcoef(df['Vehicle Age'], df['Electric Range'])[0, 1]
print(f'Pearson Correlation — Vehicle Age vs Electric Range: {corr_age_range:.4f}')
print()

print('Interpretation:')
if abs(corr_age_range) < 0.1:
    print('  Negligible correlation')
elif abs(corr_age_range) < 0.3:
    print('  Weak correlation')
elif abs(corr_age_range) < 0.5:
    print('  Moderate correlation')
else:
    print('  Strong correlation')

# Correlation matrix for all numeric cols
num_df = df[['Model Year', 'Electric Range', 'Vehicle Age', 'Utility_Complexity_Score', 'Latitude', 'Longitude']]
print()
print('=== Correlation Matrix (numeric cols) ===')
corr_matrix = num_df.corr().round(3)
print(corr_matrix.to_string())

In [ ]:
# ----- 5.7 Electric Range by Range Category (sanity check) -----
print('=== Range Stats by Range_Category ===')
print(
    df.groupby('Range_Category')['Electric Range']
    .agg(Count='count', Min='min', Mean='mean', Max='max')
    .round(1)
    .to_string()
)

In [ ]:
# ----- 5.8 Model Year × EV Type — how BEV vs PHEV share changes over time -----
print('=== Model Year × EV Type — Row % ===')
yr_type = pd.crosstab(df['Model Year'], df['Electric Vehicle Type'], normalize='index').mul(100).round(1)
# Show only years with > 100 records for clean signal
yr_counts_filter = df['Model Year'].value_counts()
valid_yrs = yr_counts_filter[yr_counts_filter >= 100].index
print(yr_type.loc[yr_type.index.isin(valid_yrs)].to_string())

---
## 6. Trend Analysis — Model Year <a id='6'></a>

In [ ]:
# ----- 6.1 Yearly Registration Count & YoY Growth -----
yr_reg = df['Model Year'].value_counts().sort_index()

# YoY growth rate
yr_df = pd.DataFrame({'Registrations': yr_reg})
yr_df['YoY_Growth_%'] = yr_df['Registrations'].pct_change().mul(100).round(1)

# Filter to years with meaningful volume (post-2010)
yr_df_filtered = yr_df[yr_df.index >= 2011]

print('=== Yearly Registration Trend ===')
print(yr_df_filtered.to_string())
print()
print(f'Peak year (registrations): {yr_df_filtered["Registrations"].idxmax()} — {yr_df_filtered["Registrations"].max():,} units')
print(f'Avg YoY growth (2012–2024): {yr_df_filtered.loc[2012:2024, "YoY_Growth_%"].mean():.1f}%')

In [ ]:
# ----- 6.2 Average Electric Range Over Time -----
print('=== Average Electric Range by Model Year (≥ 2013) ===')
range_trend = (
    df[df['Model Year'] >= 2013]
    .groupby('Model Year')['Electric Range']
    .agg(Count='count', Mean_Range='mean', Median_Range='median')
    .round(1)
)
print(range_trend.to_string())

# Overall range improvement
r2013 = range_trend.loc[2013, 'Mean_Range']
r2024 = range_trend.loc[2024, 'Mean_Range']
print()
print(f'Mean range in 2013 : {r2013:.1f} miles')
print(f'Mean range in 2024 : {r2024:.1f} miles')
print(f'Improvement        : {((r2024 - r2013) / r2013 * 100):.1f}%')

In [ ]:
# ----- 6.3 BEV Growth Over Time -----
print('=== BEV Count by Model Year ===')
bev = df[df['Electric Vehicle Type'] == 'Battery Electric Vehicle (BEV)']
phev = df[df['Electric Vehicle Type'] == 'Plug-in Hybrid Electric Vehicle (PHEV)']

bev_yr = bev['Model Year'].value_counts().sort_index().rename('BEV')
phev_yr = phev['Model Year'].value_counts().sort_index().rename('PHEV')

ev_type_yr = pd.concat([bev_yr, phev_yr], axis=1).fillna(0).astype(int)
ev_type_yr = ev_type_yr[ev_type_yr.index >= 2011]
ev_type_yr['Total'] = ev_type_yr['BEV'] + ev_type_yr['PHEV']
ev_type_yr['BEV_%'] = (ev_type_yr['BEV'] / ev_type_yr['Total'] * 100).round(1)

print(ev_type_yr.to_string())

In [ ]:
# ----- 6.4 Top Make Share Over Time (Top 5) -----
print('=== Top 5 Makes — Yearly Registration Count ===')
top5 = df['Make'].value_counts().head(5).index.tolist()
make_yr = (
    df[df['Make'].isin(top5) & (df['Model Year'] >= 2015)]
    .groupby(['Model Year', 'Make'])
    .size()
    .unstack(fill_value=0)
)
print(make_yr.to_string())

---
## 7. Geographic Concentration <a id='7'></a>

In [ ]:
# ----- 7.1 Top 15 Counties -----
print('=== Top 15 Counties by EV Registrations ===')
county_counts = df['County'].value_counts().head(15)

for county, cnt in county_counts.items():
    pct = cnt / len(df) * 100
    bar = '█' * (cnt // 500)
    print(f'  {county:<12}: {cnt:>6,}  ({pct:5.1f}%)  {bar}')

print()
top3_share = county_counts.head(3).sum() / len(df) * 100
print(f'Top 3 counties hold {top3_share:.1f}% of all registrations')

In [ ]:
# ----- 7.2 Top 15 Cities -----
print('=== Top 15 Cities by EV Registrations ===')
city_counts = df['City'].value_counts().head(15)

for city, cnt in city_counts.items():
    pct = cnt / len(df) * 100
    print(f'  {city:<20}: {cnt:>6,}  ({pct:4.1f}%)')

In [ ]:
# ----- 7.3 EV Type Distribution within Top 5 Counties -----
top5_counties = df['County'].value_counts().head(5).index

print('=== EV Type Split — Top 5 Counties ===')
county_type = (
    df[df['County'].isin(top5_counties)]
    .groupby(['County', 'Electric Vehicle Type'])
    .size()
    .unstack(fill_value=0)
)
county_type.columns = ['BEV', 'PHEV']
county_type['Total'] = county_type['BEV'] + county_type['PHEV']
county_type['BEV_%'] = (county_type['BEV'] / county_type['Total'] * 100).round(1)
print(county_type.to_string())

In [ ]:
# ----- 7.4 Geographic Spread — Lat/Long Stats -----
print('=== Geographic Bounds (Lat / Long) ===')
print(f'  Latitude  — Min: {df["Latitude"].min():.4f}, Max: {df["Latitude"].max():.4f}, Mean: {df["Latitude"].mean():.4f}')
print(f'  Longitude — Min: {df["Longitude"].min():.4f}, Max: {df["Longitude"].max():.4f}, Mean: {df["Longitude"].mean():.4f}')

# Flag out-of-Washington coordinates
# WA lat: ~45.5–49.0, lon: ~-124.7 to -116.9
out_of_wa = df[(df['Latitude'] < 45.5) | (df['Latitude'] > 49.5) |
               (df['Longitude'] < -125.0) | (df['Longitude'] > -116.5)]
print()
print(f'Records outside WA bounding box: {len(out_of_wa)}')
if len(out_of_wa) > 0:
    print(out_of_wa[['County','City','State','Latitude','Longitude']].head(10).to_string())

---
## 8. Outlier Detection <a id='8'></a>

In [ ]:
# ----- 8.1 IQR-based Outlier Detection -----
def iqr_outliers(series, name):
    q1  = np.percentile(series.dropna(), 25)
    q3  = np.percentile(series.dropna(), 75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = series[(series < lower) | (series > upper)]
    pct = len(outliers) / len(series) * 100
    print(f'{name}')
    print(f'  Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}')
    print(f'  Bounds: [{lower:.2f}, {upper:.2f}]')
    print(f'  Outliers: {len(outliers):,} ({pct:.2f}%)')
    print()
    return outliers

print('=== IQR-based Outlier Detection ===')
out_range = iqr_outliers(df['Electric Range'], 'Electric Range')
out_age   = iqr_outliers(df['Vehicle Age'],    'Vehicle Age')
out_yr    = iqr_outliers(df['Model Year'],      'Model Year')

In [ ]:
# ----- 8.2 Z-score Outlier Detection -----
def zscore_outliers(series, name, threshold=3):
    mean = series.mean()
    std  = series.std()
    z    = np.abs((series - mean) / std)
    outliers = series[z > threshold]
    pct = len(outliers) / len(series) * 100
    print(f'{name}')
    print(f'  Mean={mean:.2f}, Std={std:.2f}')
    print(f'  |Z| > {threshold} outliers: {len(outliers):,} ({pct:.2f}%)')
    print()
    return outliers

print('=== Z-Score Outlier Detection (|Z| > 3) ===')
_ = zscore_outliers(df['Electric Range'], 'Electric Range')
_ = zscore_outliers(df['Vehicle Age'],    'Vehicle Age')
_ = zscore_outliers(df['Model Year'],      'Model Year')

In [ ]:
# ----- 8.3 Profile of Electric Range Outliers (IQR method) -----
if len(out_range) > 0:
    print('=== Electric Range Outlier Profiles ===')
    outlier_df = df.loc[out_range.index]
    print(outlier_df[['Make','Model','Model Year','Electric Vehicle Type','Electric Range']]
          .sort_values('Electric Range', ascending=False)
          .head(20)
          .to_string(index=False))
else:
    print('No IQR outliers detected in Electric Range — data appears well-bounded.')

In [ ]:
# ----- 8.4 Very Old Vehicles (potential data anomalies) -----
old_thresh = 2010
old_vehicles = df[df['Model Year'] < old_thresh]
print(f'=== Vehicles with Model Year < {old_thresh} ===')
print(f'Count: {len(old_vehicles)}')
print(old_vehicles[['Make','Model','Model Year','Electric Vehicle Type','Electric Range','County']]
      .sort_values('Model Year')
      .to_string(index=False))

In [ ]:
# ----- 8.5 Zero-range PHEV/BEV Records -----
zero_range = df[df['Electric Range'] == 0]
print(f'=== Vehicles with Electric Range = 0 ===')
print(f'Count: {len(zero_range)}')
if len(zero_range) > 0:
    print(zero_range['Electric Vehicle Type'].value_counts())
    print()
    print('These are typically PHEVs where range data was not reported.')

---
## 9. Key EDA Findings Summary <a id='9'></a>

In [ ]:
# Clean up temporary column added during analysis
df.drop(columns=['Range_Bin'], inplace=True, errors='ignore')

print('Temporary columns cleaned. Dataset shape:', df.shape)

### Summary of Key EDA Findings

| # | Finding | Detail |
|---|---------|--------|
| 1 | **Dataset is complete** | 101,275 records, 26 columns, zero missing values |
| 2 | **PHEV dominates slightly** | 55,692 PHEVs (55%) vs 45,583 BEVs (45%), but BEV share is rising sharply in recent years |
| 3 | **Tesla is #1 by large margin** | 24,182 registrations (23.9%) — nearly 2.5× the next brand (Chevrolet at 9,848) |
| 4 | **Range gap between BEV and PHEV is massive** | BEV mean: 200 miles vs PHEV mean: 32 miles |
| 5 | **Range has improved over time** | Mean range grew from ~75 miles (2013) to ~160+ miles (2024) |
| 6 | **Fleet is relatively young** | Median vehicle age: 7 years; most registrations are 2017–2025 |
| 7 | **Registration peaked in 2018** | 13,956 units in 2018; next peaks in 2020 (11,970) and 2024 (10,723) |
| 8 | **Highly geographically concentrated** | King County alone = 46% of fleet; top 3 counties = ~65% |
| 9 | **Fleet is Mass Market dominated** | 64% Mass Market vs 36% Luxury — but Luxury EVs have significantly higher range |
| 10 | **Growth Wave is dominant** | 50,163 vehicles in Growth phase; Early adopters (13,468) and Mass Adoption (37,644) |
| 11 | **No IQR outliers in Electric Range** | Distribution is bimodal (PHEV vs BEV cluster) but bounded — no erroneous extremes |
| 12 | **Very few pre-2010 EVs** | Only ~50 records before 2010 — negligible but valid (early EV experiments) |

---

**Next steps →** `04_statistical_analysis.ipynb` will:
- Perform correlation & regression analysis on Electric Range vs Model Year, Vehicle Age
- Conduct hypothesis testing (BEV vs PHEV range differences, County-level adoption differences)
- Validate distributional assumptions (normality tests, variance analysis)